# 资源分配问题

In [1]:
import numpy as np
from time import time
import random as rd

## 类定义
定义node，表示节点类

In [2]:
class node():
    def __init__(self, name=None, resourse=None):
        self.name = name  # 节点名称
        self.resourse = resourse  # 节点所用资源的列表
        self.nets = set()  # 节点所属线网的序号列表


## 定义函数
### 文件信息读取
读取文件，取得节点资源信息

In [3]:
def readNodeFile(fileName):
    file = open(fileName)
    nodes = {}
    for line in file:
        name = line.split()[0]
        resources = sum(map(int, line.split()[-10:]))
        nodes[name] = node(name, resources)
    file.close()
    return nodes

读取文件，取得线网信息

In [4]:
def readNetFile(filename):
    names = []
    netId = 0
    file = open(filename)
    for line in file:
        if 's' in line.split():
            for name in names:
                nodes[name].nets.add(netId)
            netId += 1
            names.clear()
        names.append(line.split()[0])
    nodes[name].nets.add(netId)
    file.close()

### 定义分配函数allocation

In [5]:
def allocation():
    return [rd.randint(0,N_FPGA-1) for _ in nodes.values()]

### 定义写文件函数writeResult

In [6]:
def writeResult():
    buffer = ''
    for fpga in range(N_FPGA):
        buffer += 'F'+str(fpga)
        for index, node in enumerate(nodes.values()):
            if best[index] == fpga:
                buffer += node.name+' '
        buffer += '\n'
    return buffer


### 定义资源差异函数 resourseCal
计算各个板的资源

In [7]:
def resourseCal():
    return 0

### 定义板间连线计算函数 linkCal
计算FPGA板间连线的总和

In [8]:
def linkCal(ind):
    link = 0
    fpgaNets = [set() for _ in range(N_FPGA)]  # fpga网络列表
    for index, node in enumerate(nodes.values()):
        fpgaNets[ind[index]] |= node.nets  # 将节点网络添加到染色体对应位置节点对应的fpga网络上
    for index, net1 in enumerate(fpgaNets):
        for net2 in fpgaNets[index:]:
            link += len(net1 & net2)  # 连线数量为各板间连线之和
    return link


## 执行代码

设定参数

In [9]:
N_FPGA = 4 # FPGA数量
MAXITER = 10**3 # 最大循环次数
PC = 0.5	# 交叉概率
PW = 0.5	# 变异概率
N = 100  # 染色体规模

读取和打开文件

In [10]:
# 读取节点文件
nodes = readNodeFile("./contest/testdata-0/design.are")

# 读取线网文件
readNetFile("./contest/testdata-0/design.net")

# 打开结果文件
file = open("./result.res", 'w')

In [11]:
# 根据染色体的适应值，按照一定的概率，生成新一代染色体群体newpop
def screenPop():
    sumFit = np.cumsum(fitness)
    for ind in pop:
        randVal = np.random.rand()
        for i in range(N):
            if randVal < sumFit[i]:
                ind = pop[i]
                break

In [12]:
# 根据交叉概率PC，以及各染色体的适应值fitness，通过交叉的方式生成新群体

def crossPop():
    for i in range(int(N/2)): # 遍历雄性个体
        j = rd.randint(int(N/2),N-1)  # 寻找配偶
        if np.random.rand() < PC: # 求偶成功
            index = rd.randint(1,len(nodes))  # 染色体交叉点
            child1 = pop[i][:index]+pop[j][index:]
            child2 = pop[j][:index]+pop[i][index:]
            pop[i], pop[j] = child1, child2  # 生成新染色体

In [13]:
def mutatePop():
    for ind in pop:
        if rd.random() < PW:  # 如果随机数小于概率就进行变异
            gene1, gene2 = rd.sample(ind, 2)
            gene1, gene2 = gene2, gene1

In [14]:
# 步骤1，产生N个染色体的初始群体,保存在pop里面
pop = [allocation() for _ in range(N)]  # 初始化种群
tstart = time()  # 开始计时
for iter in range(MAXITER):
    links = [linkCal(ind) for ind in pop]  # 计算每个染色体的连线数量
    fitness = [1/link for link in links]  # 计算每个染色体的适应值
    maxFit, minFit = max(fitness), min(fitness)
    index1 = fitness.index(maxFit)
    index2 = fitness.index(minFit)
    best, worst = pop[index1], pop[index2]
    screenPop()  # 选取出一个新的群体
    crossPop()  # 步骤4：交叉产生新染色体，得到新群体
    mutatePop()  # 步骤5：基因变异
    pop[0] = best  # 保留上一代中适应值最高的染色体(替换掉适应值最低的染色体）
    if np.mod(iter, MAXITER/10) == 1:
        print('迭代次数：%d 运行时间：%f秒 板间连线总数：%d' %
              (iter, time()-tstart, links[index1]))
print('最优板间连线：%d 适应值：%f 分配关系：' % (links[index1], maxFit))
print(writeResult())  # 输出最优染色体/路径
print('问题耗时 :  %f 秒.' % (time()-tstart))

迭代次数：1 运行时间：0.020448秒 板间连线总数：143
迭代次数：101 运行时间：0.819950秒 板间连线总数：135
迭代次数：201 运行时间：1.631300秒 板间连线总数：127
迭代次数：301 运行时间：2.334362秒 板间连线总数：114
迭代次数：401 运行时间：3.009962秒 板间连线总数：109
迭代次数：501 运行时间：3.695759秒 板间连线总数：109
迭代次数：601 运行时间：4.413698秒 板间连线总数：109
迭代次数：701 运行时间：5.253408秒 板间连线总数：105
迭代次数：801 运行时间：6.127722秒 板间连线总数：105
迭代次数：901 运行时间：6.947877秒 板间连线总数：99
最优板间连线：85 适应值：0.011765 分配关系：
F0g1 g5 g6 g7 g8 g10 g11 g12 g13 g14 g15 g16 g17 g18 g19 g20 g21 g22 gp2 gp6 gp7 gp8 gp14 gp16 
F1gp5 gp9 gp15 
F2g9 g26 gp0 gp3 gp4 gp10 gp12 gp13 gp19 gp20 
F3g0 g2 g3 g4 g23 g24 g25 g27 g28 g29 g30 gp1 gp11 gp17 gp18 gp21 

问题耗时 :  7.618950 秒.
